### gender
    female  502
    male    498

### age_group
    21-29  	    200
    30-39	    204
    40-49	    180
    Above 50	416

### ahi_3_remapped
    Prefer not to answer 	39
    bottom         	        339
    middle	                327
    upper	                297

### education_level_remapped
    college_vocational_diploma	229
    other	                    11
    postgraduate	            160
    primary_secondary	        220
    undergraduate	            380

### race_ethnicity
    Asian or Pacific Islander	        231
    Black or African American	        184
    Hispanic or Latino	                166
    Middle Eastern or North African	    32
    White or Caucasian	                313
    Native American or Alaska Native	5
    A race/ethnicity not listed here	69

### extraversion_median
    0         360
    1         640

### agreeableness_median
    0         489
    1         511

### conscientiousness_median
    0         433
    1         567

### neuroticism_median
    0         344
    1         656

### openness_median
    0         474
    1         526

In [ ]:
import json
import os
from datetime import datetime
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from PIL import Image
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration


In [ ]:
#INITAL CONFIG

PROJECT_DIR = Path.cwd().parent

# options are: "Qwen3-VL-2B" - 
MODEL_NAME =  "Qwen3-VL-2B"

#Create output folders
try:
    os.mkdir((PROJECT_DIR / "outputs" / "inferences").resolve())
except FileExistsError:
    print("Directory already exists.")

#TODO if token not present use NONE
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [ ]:
data = pd.read_csv(f"{PROJECT_DIR}/data/specs_data_safe.csv")
print(data.shape)
data.head(1)

In [ ]:

def build_prompt_blocks(row, do_persona=True):
    persona_block = f"""Persona profile:
- Country of residence: {row["Country"]}
- Gender: {row["gender"]}
- Age group: {row["age_group"]}
- Education level: {row["education_level_remapped"]}
- Race/ethnicity: {row["race_ethnicity"]}
- Socioeconomic group: {row["ahi_3_remapped"]}

Personality Big Five (scale: 1 = very low, 5 = very high):
- Extraversion: {row["extraversion"]}
- Agreeableness: {row["agreeableness"]}
- Conscientiousness: {row["conscientiousness"]}
- Neuroticism: {row["neuroticism"]}
- Openness: {row["openness"]}

Adopt the perspective of a person with this profile. Let the profile inform how you interpret the images.
"""

    task_block = """Which place looks safer?

You are shown two street-level images of urban places, Image A and Image B. Their order carries no information.
Judge only from features visible in the images. Compare the visible characteristics of both places.
"""

    tail = """
Answer options, pick one:
- "image_A": the place in Image A looks safer.
- "image_B": the place in Image B looks safer.
- "equal": the two places look equally safe.

Provide an justification of your answer, describe which characteristics influenced your perception of safety for each image.

For each image, give exactly 3 short keywords naming visible features of that image only that affected how safe the place looks.

Return only a valid JSON object with exactly these keys, no Markdown, no code fences, no other text:

{
    "answer": "<image_A | image_B | equal>",
    "keywords_image_A": "<keyword1>, <keyword2>, <keyword3>",
    "keywords_image_B": "<keyword1>, <keyword2>, <keyword3>",
    "justification_image_A": "<visible features only of Image A that affected how safe the place looks>",
    "justification_image_B": "<visible features only of Image B that affected how safe the place looks>"
    
}
"""

    head = (persona_block + "\n" + task_block) if do_persona else task_block
    return head, tail


def parse_json_response(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        repaired = text.strip()
        if not repaired.endswith(('"', "}")):
            repaired += '"'
        if not repaired.endswith("}"):
            repaired += "}"
        return json.loads(repaired)


def resize_image(image_path, max_size=1024):
    image = Image.open(image_path).convert("RGB")
    image.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
    return image


In [ ]:
# Load Qwen3-VL-2B
if MODEL_NAME == "Qwen3-VL-2B":
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        "Qwen/Qwen3-VL-2B-Instruct",
        token=hf_token,
        dtype="auto",
        device_map="auto"
    )

    processor = AutoProcessor.from_pretrained(
        "Qwen/Qwen3-VL-2B-Instruct"
    )

    model.generation_config.do_sample = False
    model.generation_config.temperature = None
    model.generation_config.top_p = None
    model.generation_config.top_k = None

    print(model.generation_config)

In [ ]:
ANSWER_MAP_NORMAL = {
    "image_a": "left",
    "image_b": "right",
    "equal": "equal",
}

ANSWER_MAP_INVERTED = {
    "image_a": "right",
    "image_b": "left",
    "equal": "equal",
}

education_mapping = {
    "college_vocational_diploma": "Vocational college education",
    "other": "Not specified",
    "postgraduate": "Postgraduate education",
    "primary_secondary": "Primary and secondary education",
    "undergraduate": "Undergraduate education",
}

race_mapping = {
    "Asian or Pacific Islander": "Asian or Pacific Islander",
    "Black or African American": "Black or African American",
    "Hispanic or Latino": "Hispanic or Latino",
    "Middle Eastern or  North African": "Middle Eastern or North African",
    "White or Caucasian": "White or Caucasian",
    "Native American or Alaska Native": "Native American or Alaska Native",
    "A race/ethnicity not listed here": "Not specified",
}

data["education_level_remapped"] = (
    data["education_level_remapped"].map(education_mapping).fillna(data["education_level_remapped"])
)
data["race_ethnicity"] = data["race_ethnicity"].map(race_mapping).fillna(data["race_ethnicity"])

In [ ]:
results_df = pd.DataFrame(
    columns=[
        "comparison_id",
        "respondent_id",
        "question",
        "human_answer",
        "model_answer",
        "model_justification_left",
        "model_justification_right",
        "model_keywords_left",
        "model_keywords_right",
        "model_raw_response",
        "model_error",
        "left_image_path",
        "right_image_path",
    ]
)

timestamp = datetime.now().astimezone().strftime("%Y-%m-%d_%H-%M")
now = datetime.now().astimezone().strftime("%H:%M:%S - %d/%m/%Y")
INVERT_IMAGES = True
START = 0
STOP = 60
DOWNSIZE = True  # If true the images will be downsized, useful for testing the expirement pipeline
SIZE = 768 #1024s
DO_PERSONA = True  # if False model baseline if True persona profile enabled

print(
    f"Running starting at: {now}\nParameters:\n\t{DO_PERSONA = }\n\t{DOWNSIZE = }\n\t{INVERT_IMAGES = }\n\t{START = } - {STOP = }\n"
)

idx_range = range(START, STOP)


for idx, row in data.iterrows():
    if idx not in idx_range:
        continue

    if row["Question"] != "safe":
        continue

    
    raw_response = None
    stage = "init"
    model_error = None
    left_image_path = None
    right_image_path = None

    print(f"inference: {idx}/{STOP-START}")

    try:
        stage = "prompt"
        head, tail = build_prompt_blocks(row, DO_PERSONA)
        prompt = head + tail

        if idx == START:
            print(prompt)

        stage = "load_images"
        left_image_path = (PROJECT_DIR / row["left_path"]).resolve()
        right_image_path = (PROJECT_DIR / row["right_path"]).resolve()

        if DOWNSIZE is False:
            # Load native image
            left_image = Image.open(left_image_path).convert("RGB")
            right_image = Image.open(right_image_path).convert("RGB")
        else:
            # Load resized image
            left_image = resize_image(left_image_path, SIZE)
            right_image = resize_image(right_image_path, SIZE)

        # image_A = first presented image, image_B = second presented image.
        # INVERT_IMAGES flips only the presentation order. Paths and parsed
        # outputs stay in dataset identity (left/right).
        if INVERT_IMAGES:  # Used to check if the image order of appearance is influencing the model decision
            first_image, second_image = right_image, left_image
        else:
            first_image, second_image = left_image, right_image

        stage = "inference"
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": head},
                    {"type": "text", "text": "Image A:"},
                    {"type": "image", "image": first_image},
                    {"type": "text", "text": "Image B:"},
                    {"type": "image", "image": second_image},
                    {"type": "text", "text": tail},
                ],
            }
        ]
        # Prepare text and images together
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        )

        # inputs -> model device
        inputs = inputs.to(model.device)

        generated_ids = model.generate(**inputs, max_new_tokens=500, do_sample=False)

        # removing input tokens
        generated_ids_trimmed = [
            output_ids[len(input_ids) :]
            for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
        ]

        # Decode
        output_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        raw_response = output_text[0]
        print(output_text)

        stage = "parse"
        result = parse_json_response(raw_response)

        answer_map = ANSWER_MAP_INVERTED if INVERT_IMAGES else ANSWER_MAP_NORMAL
        model_answer = answer_map.get(result["answer"].strip().lower(), "invalid")

        # Map per-image outputs back to dataset identity (left/right), so all
        # columns in a row use the same reference frame as human_answer.
        if INVERT_IMAGES:
            justification_left = result["justification_image_B"]
            justification_right = result["justification_image_A"]
            keywords_left = result["keywords_image_B"]
            keywords_right = result["keywords_image_A"]
        else:
            justification_left = result["justification_image_A"]
            justification_right = result["justification_image_B"]
            keywords_left = result["keywords_image_A"]
            keywords_right = result["keywords_image_B"]

        new_row = {
            "comparison_id": idx,
            "respondent_id": row["Respondent"],
            "question": row["Question"],
            "human_answer": row["Score"],
            "model_answer": model_answer,
            "model_justification_left": justification_left,
            "model_justification_right": justification_right,
            "model_keywords_left": keywords_left.strip().lower(),
            "model_keywords_right": keywords_right.strip().lower(),
            "model_raw_response": raw_response,
            "model_error": None,
            "left_image_path": left_image_path,
            "right_image_path": right_image_path,
        }

    except Exception as exc:
        model_error = f"{stage}: {type(exc).__name__}: {exc}"
        print(f"[{idx}] ERROR ({model_error})")
        new_row = {
            "comparison_id": idx,
            "respondent_id": row["Respondent"],
            "question": row["Question"],
            "human_answer": row["Score"],
            "model_answer": "invalid" if raw_response is not None else "error",
            "model_justification_left": None,
            "model_justification_right": None,
            "model_keywords_left": None,
            "model_keywords_right": None,
            "model_raw_response": raw_response,
            "model_error": model_error,
            "left_image_path": left_image_path,
            "right_image_path": right_image_path,
        }

    results_df.loc[len(results_df)] = new_row
    results_df.to_csv(
        f"{PROJECT_DIR}/outputs/inferences/mllm_results_{MODEL_NAME}_{timestamp}.csv",
        index=False,
    )
    print(new_row)
    # RTX 2060
    # 37min - 30 inferencias - qwen-2b-vl - imagens reduzidas para 768px - prompt 1
    # 34min - 30 inferencias - qwen-2b-vl - imagens reduzidas para 768px - prompt 2
    # 75 min - 60 inferencias  - qwen-2b-vl - imagens reduzidas para 768px - prompt 7


In [ ]:
print(model.generation_config)